# Orchestrate sandbox jobs with Durable Task Scheduler

Use this walkthrough to provision Durable Task Scheduler (DTS), run the Durable Task worker in this notebook process, and coordinate Azure Container Apps sandbox jobs end to end.

## Scenario
You want a scheduler that can coordinate sandbox creation, workload staging, execution, snapshots, and optional cleanup without moving the worker into the sandbox itself.

> This lab is **DTS orchestration over ACA Sandboxes**. The worker stays outside the sandbox, and the official `az durabletask` extension owns scheduler and task hub lifecycle.


## What you'll learn
- Provision the resource group, DTS scheduler, and task hub
- Start the Durable Task worker in the notebook kernel
- Run one sandbox lifecycle orchestration end to end
- Optionally fan out a few sandbox jobs in parallel
- Inspect orchestration history in the DTS dashboard
- Clean up sandboxes and DTS resources intentionally


## At a glance
- `az durabletask` provisions the scheduler and task hub
- this notebook starts the Durable Task worker locally in the notebook kernel
- sandbox activities run through `azure.containerapps.sandbox`
- cleanup stays explicit and flag-driven


## Before you run the setup cell
- Sign in with Azure CLI: `az login`
- Install the sandbox SDKs, then optionally add DTS support if you want to run the notebook or script end to end:

  ```powershell
  python -m pip install azure-containerapps-sandbox
  python -m pip install durabletask-azuremanaged
  ```

- If you want the published wheel flow for this repo, use the detailed steps in the lab `README.md`; `durabletask-azuremanaged` remains an optional lab dependency there as well.
- On Windows, if the kernel cannot find Azure CLI, restart VS Code or set `AZURE_CLI_PATH`.
- The sample flags in the next cell enable cleanup. Review them before you run the final cleanup step.


### 0. Initialize variables and review cleanup flags

Names are derived from the lab folder and your Azure subscription so you can rerun the lab without hardcoding values. The sample settings below are convenient for a full walkthrough, but they also enable destructive cleanup.


In [ ]:
import importlib
import json
import logging
import sys
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(message)s")

lab_path = Path(globals().get("__vsc_ipynb_file__", Path.cwd() / "__file__")).resolve().parent
if str(lab_path) not in sys.path:
    sys.path.insert(0, str(lab_path))

import main as main_module
main_module = importlib.reload(main_module)

LabConfig = main_module.LabConfig
build_default_names = main_module.build_default_names
cleanup_sandboxes_by_id = main_module.cleanup_sandboxes_by_id
create_runtime = main_module.create_runtime
delete_dts_resources = main_module.delete_dts_resources
delete_resource_group_resources = main_module.delete_resource_group_resources
delete_sandbox_group_resources = main_module.delete_sandbox_group_resources
extract_sandbox_ids = main_module.extract_sandbox_ids
load_account = main_module.load_account
provision_dts_resources = main_module.provision_dts_resources
run_fan_out_workflow = main_module.run_fan_out_workflow
run_single_workflow = main_module.run_single_workflow
start_runtime = main_module.start_runtime
stop_runtime = main_module.stop_runtime

existing_runtime = globals().get("runtime")
if existing_runtime is not None:
    stop_runtime(existing_runtime)
    runtime = None

account = load_account()
lab_name = lab_path.name
defaults = build_default_names(lab_name, account["id"])

resource_group_name = defaults["resource_group"]
resource_group_location = "westus2"
scheduler_name = defaults["scheduler_name"]
task_hub_name = defaults["task_hub"]
sandbox_group_name = defaults["sandbox_group"]
scheduler_sku = "Consumption"
scheduler_capacity = 1
scheduler_ip_allowlist = "[0.0.0.0/0]"

assign_current_user_role = True
cleanup_sandboxes = True
stop_and_resume = True
fan_out_width = 5
delete_sandbox_group_when_done = True
delete_scheduler_resources = True
delete_resource_group = True

print(f"Lab:            {lab_name}")
print(f"User:           {account['user']['name']}")
print(f"Subscription:   {account['name']} ({account['id']})")
print(f"Resource Group: {resource_group_name}")
print(f"Location:       {resource_group_location}")
print(f"Scheduler:      {scheduler_name}")
print(f"Task Hub:       {task_hub_name}")
print(f"Sandbox Group:  {sandbox_group_name}")


### 1. Provision DTS and the task hub

This step creates or reuses the resource group, scheduler, and task hub together so the later cells can focus on orchestration instead of setup details.


In [ ]:
config = LabConfig(
    lab_name=lab_name,
    subscription_id=account["id"],
    resource_group=resource_group_name,
    location=resource_group_location,
    scheduler_name=scheduler_name,
    task_hub=task_hub_name,
    sandbox_group=sandbox_group_name,
    scheduler_sku=scheduler_sku,
    scheduler_capacity=scheduler_capacity,
    ip_allowlist=scheduler_ip_allowlist,
)

provisioned = provision_dts_resources(
    config,
    assign_current_user_role=assign_current_user_role,
    principal_name=account["user"]["name"],
)
print(json.dumps(provisioned, indent=2))


### 2. Start the Durable Task worker

The worker runs **here** in the notebook kernel, not inside any sandbox. Its activities call the sandbox SDK clients that do the actual sandbox work.


In [ ]:
runtime = create_runtime(config, endpoint=provisioned["endpoint"])
start_runtime(runtime)

print(f"Endpoint: {runtime.endpoint}")
print(f"Task hub: {runtime.taskhub}")
print("Worker location: this notebook process")


### 3. Run the primary sandbox workflow

The main orchestration walks one sandbox through create -> stage workload -> execute -> capture stats -> snapshot -> optional stop/resume -> optional cleanup.


In [ ]:
single_result = run_single_workflow(
    runtime.client,
    config,
    cleanup_sandboxes=cleanup_sandboxes,
    stop_and_resume=stop_and_resume,
)
print(json.dumps(single_result, indent=2))


### 4. Run a small fan-out example

This optional orchestration starts a few sandbox jobs in parallel so you can see DTS coordinate multiple activity chains at once.


In [ ]:
fan_out_result = run_fan_out_workflow(
    runtime.client,
    config,
    fan_out_width=fan_out_width,
    cleanup_sandboxes=cleanup_sandboxes,
)
print(json.dumps(fan_out_result, indent=2))


### 5. Inspect the DTS dashboard

Open the dashboard and paste in the endpoint and task hub values from this run to inspect orchestration state and instance history.


In [ ]:
print("Dashboard:", provisioned["dashboard_url"])
print("Endpoint:", runtime.endpoint)
print("Task hub:", runtime.taskhub)
print("Primary orchestration instance:", single_result["instance_id"])
print("Fan-out orchestration instance:", fan_out_result["instance_id"])


### 6. Stop the worker and clean up deliberately

The worker is always stopped. Resource deletion follows the flags from the setup cell so you can either preserve the lab for inspection or tear everything down.


In [ ]:
cleanup_summary = {}
stop_runtime(runtime)
cleanup_summary["worker"] = "stopped"

if delete_resource_group:
    cleanup_summary["resource_group"] = delete_resource_group_resources(config)
else:
    if delete_sandbox_group_when_done:
        if not cleanup_sandboxes:
            sandbox_ids = extract_sandbox_ids(single_result) + extract_sandbox_ids(fan_out_result)
            if sandbox_ids:
                cleanup_summary["sandboxes"] = cleanup_sandboxes_by_id(config, sandbox_ids)
        cleanup_summary["sandbox_group"] = delete_sandbox_group_resources(config)

    if delete_scheduler_resources:
        cleanup_summary["dts"] = delete_dts_resources(config)

    if not delete_sandbox_group_when_done and not delete_scheduler_resources:
        cleanup_summary["note"] = "No destructive cleanup requested. Resources were left in place."

print(json.dumps(cleanup_summary, indent=2))
